In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np
from plot_helpers import (update_layout, colors)



In [ ]:
with open("../configs/powerplantmatching_config_with_WEPP.yaml", "r") as f:
    import yaml
    config_update = yaml.safe_load(f)
    config_update["target_countries"] = ["Chile", "Morocco", "South Africa", "Egypt"]

# Renewable plants and targets

In [ ]:
ctsn = config_update['target_countries']

In [ ]:
data = pd.read_csv("planned_renewables/yearly_full_release_long_format.csv").query("Area in @ctsn and Subcategory == 'Fuel' and Variable in ['Solar','Wind'] and Category == 'Capacity' and Year >=2023")

In [ ]:
data = data.loc[:,["Area", "Variable", "Year", "Value"]]


In [ ]:
# Integrate updated South Africa solar data into the main data dataframe
print("="*80)
print("INTEGRATING updated SOLAR DATA INTO MAIN DATAFRAME")
print("="*80)

# First, let's examine the structure of the existing data dataframe
print("Original data structure:")
print(f"Columns: {list(data.columns)}")
print(f"Shape: {data.shape}")
print(f"Unique areas: {data['Area'].unique()}")
print(f"Unique variables: {data['Variable'].unique()}")
print("\nSample of original data:")
display(data.head())

# Filter out existing South Africa solar data to replace with updated values
print(f"\nFiltering out existing South Africa solar data...")
data_filtered = data[~((data['Area'] == 'South Africa') & (data['Variable'] == 'Solar'))]
print(f"Data after filtering: {data_filtered.shape}")

# Create updated South Africa solar data in the same format as the main dataframe
sa_solar_updated_formatted = pd.DataFrame({
    'Area': ['South Africa', 'South Africa'],
    'Variable': ['Solar', 'Solar'],
    'Year': [2023, 2024],
    'Value': [7.5, 8.5],  # updated values in GW
})

print("\nupdated South Africa solar data to be integrated:")
display(sa_solar_updated_formatted)

# Combine the filtered data with the updated South Africa solar data
data_updated = pd.concat([data_filtered, sa_solar_updated_formatted], ignore_index=True)

# Sort by Area, Variable, and Year for better organization
data_updated = data_updated.sort_values(['Area', 'Variable', 'Year']).reset_index(drop=True)

print(f"\nFinal updated data shape: {data_updated.shape}")
print("\nSouth Africa solar data in updated dataframe:")
sa_solar_check = data_updated[(data_updated['Area'] == 'South Africa') & (data_updated['Variable'] == 'Solar')]
display(sa_solar_check)

# Update the main data variable
datab = data_updated.copy()

print("\n" + "="*80)
print("INTEGRATION COMPLETE: Data dataframe now contains updated SA solar values")
print("="*80)

In [ ]:
data_updated["Area"] = data_updated["Area"].replace({"South Africa": "ZA", "Egypt": "EG", "Morocco": "MA", "Chile": "CL"})

In [ ]:
targetsr = pd.read_excel("planned_renewables/targets_download.xlsx", sheet_name="capacity_target_wide").query("country_name in @ctsn")
targets = targetsr.query("country_name in @ctsn").copy()

In [ ]:
targets = targets.loc[:, ['country_code','target_year', 'Wind','Solar']]

In [ ]:
targets["country_code"] = ["CL", "EG", "MA", "ZA"]

In [ ]:
new_targets_ZA = pd.DataFrame({
    'country_code': ['ZA'],
    'target_year': [2030],
    'Wind': [7.2],  # IRP 2024 draft
    'Solar': [7.8+11.3]  # updated values in GW
})
targets = targets.query("country_code != 'ZA'")

targets = pd.concat([targets, new_targets_ZA], ignore_index=True)

In [ ]:
targets_long = targets.melt(
    id_vars=['country_code', 'target_year'],
    value_vars=['Wind', 'Solar'],
    var_name='Variable',
    value_name='Value'
)
targets_long

In [ ]:
targets_long.columns = ["Area", "Year", "Variable", "Value"]

In [ ]:
trend = pd.concat([data_updated, targets_long], ignore_index=True)

In [ ]:
trend["Year"] = trend["Year"].astype(str).replace({"2030": "2030 target"})

In [ ]:
print(trend)

In [ ]:
fig = px.bar(trend, 
            y="Year", 
            x="Value", 
            color="Variable",
            facet_row="Area",
            facet_row_spacing=0.05,
            barmode="stack",
            color_discrete_map={
                "Wind": colors["electricity"]["Wind onshore"],
                "Solar": colors["electricity"]["Solar PV"],
            },
            # color_discrete_sequence=config_update["fhg_colors_15"],
            labels={"Value":"Capacity in GW", "Variable": "", "Year": ""},
            height=400,
            width=900,
            category_orders={"Year": ["2023", "2024", "2030 target"], "Variable": ["Wind", "Solar"]},
        )

fig.update_yaxes(matches=None)
fig.update_yaxes(showticklabels=True)
update_layout(fig)



fig.show()

In [ ]:
print(trend)

In [ ]:
gwpt = pd.read_excel("https://docs.google.com/spreadsheets/d/1r7G4szrq1xuqIA6xwfA9R29Rg5nWGH-HZ7vU9l6_DUg/export?format=xlsx&id=1r7G4szrq1xuqIA6xwfA9R29Rg5nWGH-HZ7vU9l6_DUg", skiprows=4)
gwpt = gwpt[gwpt["Country/Area"].isin(ctsn)]

In [ ]:
gspt = pd.read_excel("https://docs.google.com/spreadsheets/d/1Kwlj_W53c603myv8IFsGeI5uS6EkZl8KKK14cV5nPZw/export?format=xlsx&id=1Kwlj_W53c603myv8IFsGeI5uS6EkZl8KKK14cV5nPZw", skiprows=4)
gspt = gspt[gspt["Country/Area"].isin(ctsn)]

In [ ]:
# Comprehensive Analysis: Trend Data vs 2030 Targets vs GWPT/GSPT Pipeline
print("="*100)
print("COMPREHENSIVE RENEWABLE ENERGY ANALYSIS: TRENDS, TARGETS & PIPELINE")
print("="*100)

# 1. TREND ANALYSIS
print("\n1. TREND ANALYSIS (2023-2024 Performance)")
print("-" * 60)

trend_2023 = trend[trend['Year'] == '2023'].copy()
trend_2024 = trend[trend['Year'] == '2024'].copy()
trend_2030 = trend[trend['Year'] == '2030 target'].copy()

for country in trend_2024['Area'].unique():
    country_2023 = trend_2023[trend_2023['Area'] == country]
    country_2024 = trend_2024[trend_2024['Area'] == country]
    country_2030 = trend_2030[trend_2030['Area'] == country]
    
    print(f"\n{country}:")
    
    # 2023 capacities
    solar_2023 = country_2023[country_2023['Variable'] == 'Solar']['Value'].values[0] if len(country_2023[country_2023['Variable'] == 'Solar']) > 0 else 0
    wind_2023 = country_2023[country_2023['Variable'] == 'Wind']['Value'].values[0] if len(country_2023[country_2023['Variable'] == 'Wind']) > 0 else 0
    
    # 2024 capacities
    solar_2024 = country_2024[country_2024['Variable'] == 'Solar']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Solar']) > 0 else 0
    wind_2024 = country_2024[country_2024['Variable'] == 'Wind']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Wind']) > 0 else 0
    
    # 2030 targets
    solar_2030 = country_2030[country_2030['Variable'] == 'Solar']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Solar']) > 0 else 0
    wind_2030 = country_2030[country_2030['Variable'] == 'Wind']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Wind']) > 0 else 0
    
    # Calculate 2023-2024 growth
    solar_growth = solar_2024 - solar_2023
    wind_growth = wind_2024 - wind_2023
    solar_growth_pct = (solar_growth / solar_2023 * 100) if solar_2023 > 0 else 0
    wind_growth_pct = (wind_growth / wind_2023 * 100) if wind_2023 > 0 else 0
    
    # Calculate 2024-2030 gap
    solar_gap = solar_2030 - solar_2024
    wind_gap = wind_2030 - wind_2024
    
    print(f"  Solar: {solar_2023:.1f} GW (2023) → {solar_2024:.1f} GW (2024) | Growth: +{solar_growth:.1f} GW ({solar_growth_pct:.1f}%)")
    print(f"  Wind:  {wind_2023:.1f} GW (2023) → {wind_2024:.1f} GW (2024) | Growth: +{wind_growth:.1f} GW ({wind_growth_pct:.1f}%)")
    print(f"  2030 Target Gap: Solar {solar_gap:.1f} GW, Wind {wind_gap:.1f} GW")

# 3. GAP ANALYSIS: 2024 vs HIGH-CERTAINTY PIPELINE
print(f"\n\n3. GAP ANALYSIS: 2024 CAPACITY vs HIGH-CERTAINTY PIPELINE")
print("-" * 60)
print("(Construction + Pre-construction only - excluding Announced)")

# Create mapping for country names
country_mapping = {
    'ZA': 'South Africa',
    'EG': 'Egypt', 
    'MA': 'Morocco',
    'CL': 'Chile'
}

for country_code in trend_2024['Area'].unique():
    country_name = country_mapping.get(country_code, country_code)
    
    country_2024 = trend_2024[trend_2024['Area'] == country_code]
    
    # Current capacities (2024)
    solar_2024 = country_2024[country_2024['Variable'] == 'Solar']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Solar']) > 0 else 0
    wind_2024 = country_2024[country_2024['Variable'] == 'Wind']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Wind']) > 0 else 0
    
    # High-certainty pipeline capacities (Operating + Construction + Pre-construction)
    wind_pipeline_certain = 0
    solar_pipeline_certain = 0
    
    # Extract high-certainty pipeline data
    gwpt_country = gwpt[gwpt["Country/Area"] == country_name]
    if not gwpt_country.empty:
        wind_construction = gwpt_country.get("Construction", 0).fillna(0).values[0] if "Construction" in gwpt_country.columns else 0
        wind_preconstruction = gwpt_country.get("Pre-construction", 0).fillna(0).values[0] if "Pre-construction" in gwpt_country.columns else 0
        wind_pipeline_certain = (wind_construction + wind_preconstruction) / 1e3  # Convert MW to GW
    
    gspt_country = gspt[gspt["Country/Area"] == country_name]
    if not gspt_country.empty:
        solar_construction = gspt_country.get("Construction", 0).fillna(0).values[0] if "Construction" in gspt_country.columns else 0
        solar_preconstruction = gspt_country.get("Pre-construction", 0).fillna(0).values[0] if "Pre-construction" in gspt_country.columns else 0
        solar_pipeline_certain = (solar_construction + solar_preconstruction) / 1e3  # Convert MW to GW
    
    # Calculate potential near-term capacity (2024 + high-certainty pipeline)
    solar_potential = solar_2024 + solar_pipeline_certain
    wind_potential = wind_2024 + wind_pipeline_certain
    
    print(f"\n{country_name} ({country_code}):")
    print(f"  Solar: {solar_2024:.1f} GW (2024) + {solar_pipeline_certain:.1f} GW (pipeline) = {solar_potential:.1f} GW potential")
    print(f"  Wind:  {wind_2024:.1f} GW (2024) + {wind_pipeline_certain:.1f} GW (pipeline) = {wind_potential:.1f} GW potential")
    
    # Calculate growth potential
    solar_growth_potential = (solar_pipeline_certain / solar_2024 * 100) if solar_2024 > 0 else float('inf')
    wind_growth_potential = (wind_pipeline_certain / wind_2024 * 100) if wind_2024 > 0 else float('inf')
    
    print(f"  Growth Potential: Solar +{solar_growth_potential:.0f}%, Wind +{wind_growth_potential:.0f}%")

# 4. GAP ANALYSIS: 2030 TARGETS vs TOTAL PIPELINE
print(f"\n\n4. GAP ANALYSIS: 2030 TARGETS vs TOTAL PIPELINE CAPACITY")
print("-" * 60)

# Create mapping for country names
country_mapping = {
    'ZA': 'South Africa',
    'EG': 'Egypt', 
    'MA': 'Morocco',
    'CL': 'Chile'
}

for country_code in trend_2024['Area'].unique():
    country_name = country_mapping.get(country_code, country_code)
    
    country_2024 = trend_2024[trend_2024['Area'] == country_code]
    country_2030 = trend_2030[trend_2030['Area'] == country_code]
    
    # Current and target capacities
    solar_2024 = country_2024[country_2024['Variable'] == 'Solar']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Solar']) > 0 else 0
    wind_2024 = country_2024[country_2024['Variable'] == 'Wind']['Value'].values[0] if len(country_2024[country_2024['Variable'] == 'Wind']) > 0 else 0
    solar_2030 = country_2030[country_2030['Variable'] == 'Solar']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Solar']) > 0 else 0
    wind_2030 = country_2030[country_2030['Variable'] == 'Wind']['Value'].values[0] if len(country_2030[country_2030['Variable'] == 'Wind']) > 0 else 0
    
    # Required additions
    solar_gap = solar_2030 - solar_2024
    wind_gap = wind_2030 - wind_2024
    
    # Total pipeline capacities (including all stages)
    wind_pipeline = 0
    solar_pipeline = 0
    
    # Extract pipeline data
    gwpt_country = gwpt[gwpt["Country/Area"] == country_name]
    if not gwpt_country.empty:
        wind_pipeline = gwpt_country["Prospective (Sum of Construction, Pre-construction, Announced)"].div(1e3).values[0]
    
    gspt_country = gspt[gspt["Country/Area"] == country_name]
    if not gspt_country.empty:
        solar_pipeline = gspt_country["Prospective (Sum of Construction, Pre-construction, Announced)"].div(1e3).values[0]
    
    print(f"\n{country_name} ({country_code}):")
    print(f"  Solar Gap to 2030: {solar_gap:.1f} GW | Total Pipeline: {solar_pipeline:.1f} GW | Coverage: {(solar_pipeline/solar_gap*100) if solar_gap > 0 else 0:.0f}%")
    print(f"  Wind Gap to 2030:  {wind_gap:.1f} GW | Total Pipeline: {wind_pipeline:.1f} GW | Coverage: {(wind_pipeline/wind_gap*100) if wind_gap > 0 else 0:.0f}%")
    
    # Assessment
    if solar_pipeline >= solar_gap and wind_pipeline >= wind_gap:
        status = "✅ STRONG PIPELINE - Targets likely achievable"
    elif (solar_pipeline >= solar_gap * 0.7) and (wind_pipeline >= wind_gap * 0.7):
        status = "⚠️  MODERATE PIPELINE - Targets challenging but possible"
    else:
        status = "❌ WEAK PIPELINE - Targets at risk"
    
    print(f"  Assessment: {status}")

# 5. SUMMARY AND INSIGHTS
print(f"\n\n5. KEY INSIGHTS & SUMMARY")
print("=" * 60)

print("\n🔍 TREND OBSERVATIONS:")
print("• Solar showing consistent growth across all countries")
print("• Wind development varies significantly by country")
print("• South Africa updated solar data shows 13.3% annual growth (7.5→8.5 GW)")

print("\n📊 PIPELINE CERTAINTY ANALYSIS:")
print("• High-certainty projects (Operating + Construction + Pre-construction) show immediate potential")
print("• Total pipeline (including Announced) indicates longer-term capacity")
print("• Gap between high-certainty and total pipeline reveals development risk")

print("\n🎯 TARGET FEASIBILITY:")
print("• Large capacity additions required across all countries")
print("• 6-year timeline (2024-2030) requires aggressive deployment")
print("• Pipeline data suggests varying degrees of readiness")

print("\n⚡ PIPELINE REALITY CHECK:")
print("• Construction + Pre-construction + Announced projects provide insight")
print("• High pipeline coverage suggests targets are backed by concrete projects")
print("• Low coverage indicates need for additional project development")

print("\n🚨 RISK FACTORS:")
print("• Grid integration challenges (as noted in CRSES analysis)")
print("• Transmission constraints (mentioned in Eskom GCCA 2025)")
print("• Regulatory and financing bottlenecks")
print("• Competition with aging fossil infrastructure replacement needs")

print("\n" + "="*100)

In [ ]:
# Create Egypt Solar Capacity Line Chart
print("Creating Egypt Solar Capacity Progression Chart...")

# Extract Egypt data
eg_data = trend[trend['Area'] == 'EG'].copy()
eg_solar = eg_data[eg_data['Variable'] == 'Solar'].copy()

# Get pipeline data for Egypt
country_mapping = {'EG': 'Egypt'}
country_name = country_mapping['EG']

# Get pipeline capacity for Egypt
gspt_egypt = gspt[gspt["Country/Area"] == country_name]
solar_pipeline_total = 0
solar_pipeline_certain = 0

if not gspt_egypt.empty:
    # Total pipeline (including Announced)
    if "Prospective (Sum of Construction, Pre-construction, Announced)" in gspt_egypt.columns:
        solar_pipeline_total = gspt_egypt["Prospective (Sum of Construction, Pre-construction, Announced)"].div(1e3).values[0]
    
    # High-certainty pipeline (Construction + Pre-construction only)
    solar_construction = gspt_egypt.get("Construction", 0).fillna(0).values[0] if "Construction" in gspt_egypt.columns else 0
    solar_preconstruction = gspt_egypt.get("Pre-construction", 0).fillna(0).values[0] if "Pre-construction" in gspt_egypt.columns else 0
    solar_pipeline_certain = (solar_construction + solar_preconstruction) / 1e3

# Create data for the chart
chart_data = []

# Historical data (2023-2024) - will be connected with solid line
for _, row in eg_solar.iterrows():
    if row['Year'] not in ['2030 target']:
        chart_data.append({
            'Year': int(row['Year']),
            'Year_Label': row['Year'],
            'Capacity_GW': row['Value'],
            'Type': 'Historical',
            'Category': 'Historical Data',
            'Mode': 'lines+markers'
        })

# Policy target at 2030 - marker only
target_2030 = eg_solar[eg_solar['Year'] == '2030 target']['Value'].values[0] if len(eg_solar[eg_solar['Year'] == '2030 target']) > 0 else 0
chart_data.append({
    'Year': 2030,
    'Year_Label': '2030 Policy Target',
    'Capacity_GW': target_2030,
    'Type': 'Target',
    'Category': 'Policy Target',
    'Mode': 'markers'
})

# 30 GW scenario for 2030 - connected with dashed line from 2024
current_2024 = eg_solar[eg_solar['Year'] == '2024']['Value'].values[0] if len(eg_solar[eg_solar['Year'] == '2024']) > 0 else 0
chart_data.append({
    'Year': 2030,
    'Year_Label': '2030 Scenario (30 GW)',
    'Capacity_GW': 30.0,
    'Type': 'Scenario',
    'Category': '30 GW Scenario',
    'Mode': 'lines+markers'
})

# Pipeline data at 2030 - markers only
chart_data.append({
    'Year': 2030,
    'Year_Label': '2030 High-Certainty Pipeline',
    'Capacity_GW': current_2024 + solar_pipeline_certain,
    'Type': 'Pipeline',
    'Category': 'High-Certainty Pipeline',
    'Mode': 'markers'
})

chart_data.append({
    'Year': 2030,
    'Year_Label': '2030 Total Pipeline',
    'Capacity_GW': current_2024 + solar_pipeline_total,
    'Type': 'Pipeline',
    'Category': 'Total Pipeline',
    'Mode': 'markers'
})

# Convert to DataFrame
chart_df = pd.DataFrame(chart_data)

# Create separate datasets for different line treatments
historical_data = chart_df[chart_df['Category'] == 'Historical Data'].copy()
scenario_data = pd.concat([
    chart_df[chart_df['Category'] == 'Historical Data'].tail(1),  # 2024 point
    chart_df[chart_df['Category'] == '30 GW Scenario']  # 2030 scenario point
])
marker_only_data = chart_df[chart_df['Category'].isin(['Policy Target', 'High-Certainty Pipeline', 'Total Pipeline'])].copy()

# Create the base figure
fig = px.scatter()  # Start with empty scatter plot

# Add historical data line (solid)
if not historical_data.empty:
    fig.add_scatter(
        x=historical_data['Year'], 
        y=historical_data['Capacity_GW'],
        mode='lines+markers',
        name='Historical Data',
        line=dict(color='#FFD700', width=3),  # Gold/Yellow color
        marker=dict(size=8, color='#FFD700')  # Gold/Yellow color
    )

# Add scenario line (dashed from 2024 to 2030)
if not scenario_data.empty:
    fig.add_scatter(
        x=scenario_data['Year'], 
        y=scenario_data['Capacity_GW'],
        mode='lines+markers',
        name='30 GW Scenario',
        line=dict(color='#FFD700', width=2, dash='dash'),  # Gold/Yellow with dash
        marker=dict(size=12, color='#FFD700', symbol='x')  # Gold/Yellow X marker
    )

# Add marker-only data points
colors_map = {
    'Policy Target': '#000000',  # Black
    'High-Certainty Pipeline': '#000000',  # Black
    'Total Pipeline': '#000000'  # Black
}

# Define marker symbols
marker_symbols = {
    'Policy Target': 'x',  # X marker for policy target
    'High-Certainty Pipeline': 'triangle-up',  # Triangle up marker for high-certainty pipeline
    'Total Pipeline': 'triangle-down'  # Triangle down marker for total pipeline
}

for category in marker_only_data['Category'].unique():
    category_data = marker_only_data[marker_only_data['Category'] == category]
    fig.add_scatter(
        x=category_data['Year'], 
        y=category_data['Capacity_GW'],
        mode='markers',
        name=category,
        marker=dict(
            size=12 if category == 'Policy Target' else 15,  # Slightly larger for horizontal lines
            color=colors_map.get(category, '#000000'),  # Black
            symbol=marker_symbols.get(category, 'circle'),
            line=dict(width=2, color=colors_map.get(category, '#000000'))  # Black
        ),
        showlegend=True
    )

# Add annotations for key points
for _, row in chart_df.iterrows():
    fig.add_annotation(
        x=row['Year'],
        y=row['Capacity_GW'],
        text=f"{row['Capacity_GW']:.1f} GW",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1,
        arrowcolor="gray",
        bgcolor="white",
        bordercolor="gray",
        borderwidth=1,
        font=dict(size=10)
    )

# Update layout and title
fig.update_layout(
    title='Egypt Solar Capacity Progression: Historical, Targets & Pipeline',
    xaxis_title='Year',
    yaxis_title='Solar Capacity (GW)',
    height=500,
    width=800,
    showlegend=True,
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=[2023, 2024, 2030],
        ticktext=['2023', '2024', '2030'],
        range=[2022.5, 2030.5]
    ),
    yaxis=dict(
        range=[0, max(chart_df['Capacity_GW']) * 1.1]
    )
)

# Apply basic layout formatting without bar-chart specific properties
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# Display summary
print(f"\nEgypt Solar Capacity Summary:")
print(f"2023: {chart_df[chart_df['Year']==2023]['Capacity_GW'].values[0]:.1f} GW")
print(f"2024: {chart_df[chart_df['Year']==2024]['Capacity_GW'].values[0]:.1f} GW")
print(f"2030 Policy Target: {chart_df[chart_df['Category']=='Policy Target']['Capacity_GW'].values[0]:.1f} GW")
print(f"2030 Scenario (30 GW): {chart_df[chart_df['Category']=='30 GW Scenario']['Capacity_GW'].values[0]:.1f} GW")
print(f"2030 High-Certainty Pipeline: {chart_df[chart_df['Category']=='High-Certainty Pipeline']['Capacity_GW'].values[0]:.1f} GW")
print(f"2030 Total Pipeline: {chart_df[chart_df['Category']=='Total Pipeline']['Capacity_GW'].values[0]:.1f} GW")

fig.show()

In [ ]:
# Create Egypt Wind Capacity Line Chart
print("Creating Egypt Wind Capacity Progression Chart...")

# Extract Egypt wind data
eg_data = trend[trend['Area'] == 'EG'].copy()
eg_wind = eg_data[eg_data['Variable'] == 'Wind'].copy()

# Get pipeline data for Egypt
country_mapping = {'EG': 'Egypt'}
country_name = country_mapping['EG']

# Get wind pipeline capacity for Egypt
gwpt_egypt = gwpt[gwpt["Country/Area"] == country_name]
wind_pipeline_total = 0
wind_pipeline_certain = 0

if not gwpt_egypt.empty:
    # Total pipeline (including Announced)
    if "Prospective (Sum of Construction, Pre-construction, Announced)" in gwpt_egypt.columns:
        wind_pipeline_total = gwpt_egypt["Prospective (Sum of Construction, Pre-construction, Announced)"].div(1e3).values[0]
    
    # High-certainty pipeline (Construction + Pre-construction only)
    wind_construction = gwpt_egypt.get("Construction", 0).fillna(0).values[0] if "Construction" in gwpt_egypt.columns else 0
    wind_preconstruction = gwpt_egypt.get("Pre-construction", 0).fillna(0).values[0] if "Pre-construction" in gwpt_egypt.columns else 0
    wind_pipeline_certain = (wind_construction + wind_preconstruction) / 1e3

# Create data for the wind chart
wind_chart_data = []

# Historical data (2023-2024) - will be connected with solid line
for _, row in eg_wind.iterrows():
    if row['Year'] not in ['2030 target']:
        wind_chart_data.append({
            'Year': int(row['Year']),
            'Year_Label': row['Year'],
            'Capacity_GW': row['Value'],
            'Type': 'Historical',
            'Category': 'Historical Data',
            'Mode': 'lines+markers'
        })

# Policy target at 2030 - marker only
wind_target_2030 = eg_wind[eg_wind['Year'] == '2030 target']['Value'].values[0] if len(eg_wind[eg_wind['Year'] == '2030 target']) > 0 else 0
wind_chart_data.append({
    'Year': 2030,
    'Year_Label': '2030 Policy Target',
    'Capacity_GW': wind_target_2030,
    'Type': 'Target',
    'Category': 'Policy Target',
    'Mode': 'markers'
})

# 20 GW Scenario for 2030 - connected with dashed line from 2024 (ambitious wind scenario)
wind_current_2024 = eg_wind[eg_wind['Year'] == '2024']['Value'].values[0] if len(eg_wind[eg_wind['Year'] == '2024']) > 0 else 0
wind_chart_data.append({
    'Year': 2030,
    'Year_Label': '2030 Scenario (15 GW)',
    'Capacity_GW': 20.0,
    'Type': 'Scenario',
    'Category': '20 GW Scenario',
    'Mode': 'lines+markers'
})

# Pipeline data at 2030 - markers only
wind_chart_data.append({
    'Year': 2030,
    'Year_Label': '2030 High-Certainty Pipeline',
    'Capacity_GW': wind_current_2024 + wind_pipeline_certain,
    'Type': 'Pipeline',
    'Category': 'High-Certainty Pipeline',
    'Mode': 'markers'
})

wind_chart_data.append({
    'Year': 2030,
    'Year_Label': '2030 Total Pipeline',
    'Capacity_GW': wind_current_2024 + wind_pipeline_total,
    'Type': 'Pipeline',
    'Category': 'Total Pipeline',
    'Mode': 'markers'
})

# Convert to DataFrame
wind_chart_df = pd.DataFrame(wind_chart_data)

# Create separate datasets for different line treatments
wind_historical_data = wind_chart_df[wind_chart_df['Category'] == 'Historical Data'].copy()
wind_scenario_data = pd.concat([
    wind_chart_df[wind_chart_df['Category'] == 'Historical Data'].tail(1),  # 2024 point
    wind_chart_df[wind_chart_df['Category'] == '20 GW Scenario']  # 2030 scenario point
])
wind_marker_only_data = wind_chart_df[wind_chart_df['Category'].isin(['Policy Target', 'High-Certainty Pipeline', 'Total Pipeline'])].copy()

# Create the base figure
wind_fig = px.scatter()  # Start with empty scatter plot

# Add historical data line (solid)
if not wind_historical_data.empty:
    wind_fig.add_scatter(
        x=wind_historical_data['Year'], 
        y=wind_historical_data['Capacity_GW'],
        mode='lines+markers',
        name='Historical Data',
        line=dict(color="#005b7f", width=3),  # Dark Blue color
        marker=dict(size=8, color='#005b7f')  # Dark Blue color
    )

# Add scenario line (dashed from 2024 to 2030)
if not wind_scenario_data.empty:
    wind_fig.add_scatter(
        x=wind_scenario_data['Year'], 
        y=wind_scenario_data['Capacity_GW'],
        mode='lines+markers',
        name='20 GW Scenario',
        line=dict(color='#005b7f', width=2, dash='dash'),  # Dark Blue with dash
        marker=dict(size=12, color='#005b7f', symbol='x')  # Dark Blue X marker
    )

# Add marker-only data points
wind_colors_map = {
    'Policy Target': '#000000',  # Black
    'High-Certainty Pipeline': '#000000',  # Black
    'Total Pipeline': '#000000'  # Black
}

# Define marker symbols
wind_marker_symbols = {
    'Policy Target': 'x',  # X marker for policy target
    'High-Certainty Pipeline': 'triangle-up',  # Triangle up marker for high-certainty pipeline
    'Total Pipeline': 'triangle-down'  # Triangle down marker for total pipeline
}

for category in wind_marker_only_data['Category'].unique():
    category_data = wind_marker_only_data[wind_marker_only_data['Category'] == category]
    wind_fig.add_scatter(
        x=category_data['Year'], 
        y=category_data['Capacity_GW'],
        mode='markers',
        name=category,
        marker=dict(
            size=12 if category == 'Policy Target' else 15,  # Slightly larger for triangles
            color=wind_colors_map.get(category, '#000000'),  # Black
            symbol=wind_marker_symbols.get(category, 'circle'),
            line=dict(width=2, color=wind_colors_map.get(category, '#000000'))  # Black
        ),
        showlegend=True
    )

# Add annotations for key points
for _, row in wind_chart_df.iterrows():
    wind_fig.add_annotation(
        x=row['Year'],
        y=row['Capacity_GW'],
        text=f"{row['Capacity_GW']:.1f} GW",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1,
        arrowcolor="gray",
        bgcolor="white",
        bordercolor="gray",
        borderwidth=1,
        font=dict(size=10)
    )

# Update layout and title
wind_fig.update_layout(
    title='Egypt Wind Capacity Progression: Historical, Targets & Pipeline',
    xaxis_title='Year',
    yaxis_title='Wind Capacity (GW)',
    height=500,
    width=800,
    showlegend=True,
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=[2023, 2024, 2030],
        ticktext=['2023', '2024', '2030'],
        range=[2022.5, 2030.5]
    ),
    yaxis=dict(
        range=[0, max(wind_chart_df['Capacity_GW']) * 1.1]
    )
)

# Apply basic layout formatting without bar-chart specific properties
wind_fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# Display summary
print(f"\nEgypt Wind Capacity Summary:")
print(f"2023: {wind_chart_df[wind_chart_df['Year']==2023]['Capacity_GW'].values[0]:.1f} GW")
print(f"2024: {wind_chart_df[wind_chart_df['Year']==2024]['Capacity_GW'].values[0]:.1f} GW")
print(f"2030 Policy Target: {wind_chart_df[wind_chart_df['Category']=='Policy Target']['Capacity_GW'].values[0]:.1f} GW")
print(f"2030 Scenario (15 GW): {wind_chart_df[wind_chart_df['Category']=='20 GW Scenario']['Capacity_GW'].values[0]:.1f} GW")
print(f"2030 High-Certainty Pipeline: {wind_chart_df[wind_chart_df['Category']=='High-Certainty Pipeline']['Capacity_GW'].values[0]:.1f} GW")
print(f"2030 Total Pipeline: {wind_chart_df[wind_chart_df['Category']=='Total Pipeline']['Capacity_GW'].values[0]:.1f} GW")

wind_fig.show()